# 02 — Classical routers

**Week 2, Task 4 — prepared for the published-sample Colab run.** All code cells are intentionally unexecuted. Run top to bottom after the Task 4 source files are published, then save aggregate outputs and copy the results into [the classical-router note](../results/model_scores/week2_classical.md).

The router predicts a *rescue*: `y_router = 1` only when PyMatching is wrong and correlated matching is correct. Four models share the same training features and target, and all comparisons use the same published validation sample. The 0.5 routing threshold below is **provisional**. No test split is requested. See the [task guide](../docs/WEEK_2_TASK_GUIDE.md) and [experiment protocol](../docs/EXPERIMENT_PROTOCOL.md).

## 1. Load project source and dependencies

The Colab setup resolves `SOURCE_REF` to an exact repository commit and prints it. The Task 4 changes in `src/router/evaluate.py` must be published before a fresh Colab run; change `SOURCE_REF` to that commit for a fixed rerun. The notebook downloads only the files needed for this analysis into a separate Colab folder. It does not write into the repository.

In [ ]:
%pip install -q --disable-pip-version-check datasets numpy pandas PyYAML scikit-learn xgboost tqdm

In [ ]:
import importlib.metadata
import json
import os
import sys
from pathlib import Path
from urllib.request import urlopen

REPO = "rifahnanjiba02-arch/selective-decoder-routing"
SOURCE_REF = "main"  # Replace with the published Task 4 commit for a fixed rerun.
ROOT = Path(os.environ.get("SDR_NOTEBOOK_ROOT", "/content/selective_decoder_week2_task4"))
REQUIRED_FILES = (
    "configs/data.yaml", "configs/features.yaml", "configs/models.yaml",
    "src/router/__init__.py", "src/router/data.py", "src/router/features.py",
    "src/router/decoders.py", "src/router/labels.py", "src/router/routing.py",
    "src/router/evaluate.py", "src/router/models/__init__.py",
    "src/router/models/classical.py",
)
with urlopen(f"https://api.github.com/repos/{REPO}/commits/{SOURCE_REF}") as response:
    repository_commit = json.load(response)["sha"]
for relative_path in REQUIRED_FILES:
    destination = ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(f"https://raw.githubusercontent.com/{REPO}/{repository_commit}/{relative_path}") as response:
        destination.write_bytes(response.read())
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
try:
    from router.evaluate import rescue_metrics, system_metrics
except ImportError as exc:
    raise RuntimeError(
        "SOURCE_REF must include the published Week 2 Task 4 metric code; "
        "publish these source edits or choose its commit before running in Colab."
    ) from exc
print(f"Loaded {len(REQUIRED_FILES)} project files from {repository_commit}")

## 2. Pin data and record feature/model settings

Notebook 01 used dataset revision `9c51b27ffc51ed7ff3ef3a72167ebf4862d1f2f3`, independent samples of 50,000 train and 10,000 validation rows, and seed 42 for each named split. Reuse its 33 feature columns, including 26 activity slots; the final measurement step is included. `extract_features()` masks padded syndrome positions and reads only seven allowed source fields. Labels and decoder predictions stay in a separate target/evaluation path.

In [ ]:
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from router.features import (
    ACTIVITY_COLUMNS, FEATURE_COLUMNS, MAX_TIME_STEPS, SOURCE_FIELDS,
    extract_features,
)
from router.models.classical import (
    FORBIDDEN_COLUMNS, MODEL_NAMES, fit_router, load_model_config,
    model_settings, predict_rescue_proba, validate_features,
)

DATASET_REVISION = "9c51b27ffc51ed7ff3ef3a72167ebf4862d1f2f3"
THRESHOLD = 0.5  # Provisional diagnostic only; not an operating threshold.
with open(ROOT / "configs/data.yaml", encoding="utf-8") as file:
    data_config = yaml.safe_load(file)
with open(ROOT / "configs/features.yaml", encoding="utf-8") as file:
    feature_config = yaml.safe_load(file)
model_config = load_model_config(ROOT / "configs/models.yaml")
assert data_config["dataset_name"] == "Bauxitiego/surface-code-syndromes"
assert data_config.get("dataset_revision") in (None, DATASET_REVISION)
assert feature_config["max_time_steps"] == MAX_TIME_STEPS == 26
assert tuple(feature_config["source_fields"]) == SOURCE_FIELDS
assert feature_config["padding_mask"] == "round_widths"
expected_columns = tuple(
    column
    for candidate in feature_config["compact_features"]
    for column in (ACTIVITY_COLUMNS if candidate == "activity_across_rounds" else (candidate,))
)
assert expected_columns == FEATURE_COLUMNS and len(FEATURE_COLUMNS) == 33
assert not (set(FEATURE_COLUMNS) & FORBIDDEN_COLUMNS)
assert tuple(MODEL_NAMES) == (
    "dummy", "logistic_regression", "random_forest", "xgboost"
)
pinned_config = dict(data_config)
pinned_config["dataset_revision"] = DATASET_REVISION
config_path = ROOT / "configs/data_colab_pinned.yaml"
config_path.write_text(yaml.safe_dump(pinned_config, sort_keys=False), encoding="utf-8")
SAMPLE_SIZES = {
    "train": pinned_config["development"]["train_size"],
    "validation": pinned_config["development"]["validation_size"],
}
SEED = pinned_config["sampling_seed"]
assert SAMPLE_SIZES == {"train": 50_000, "validation": 10_000} and SEED == 42
run_settings = {
    "repository_commit": repository_commit,
    "dataset": pinned_config["dataset_name"],
    "dataset_revision": DATASET_REVISION,
    "published_splits": tuple(SAMPLE_SIZES),
    "sample_sizes": SAMPLE_SIZES,
    "sampling_seed_each_split": SEED,
    "published_split_seed": pinned_config["source_split_seed"],
    "source_fields": SOURCE_FIELDS,
    "feature_columns": FEATURE_COLUMNS,
    "max_time_steps": MAX_TIME_STEPS,
    "provisional_threshold": THRESHOLD,
    "model_settings": model_settings(model_config),
    "package_versions": {
        name: importlib.metadata.version(name)
        for name in ("datasets", "numpy", "pandas", "PyYAML", "scikit-learn", "xgboost", "tqdm")
    },
}
for name, value in run_settings.items():
    print(f"{name}: {value}")

### Feature definitions reused from notebook 01

| Output | Definition |
| --- | --- |
| `detection_event_count` | Number of 1s in real syndrome positions. |
| `event_density` | Event count divided by the sum of real positions; 0 when the denominator is zero. |
| `activity_round_00` through `activity_round_25` | Event count at each time step, including the final measurement step; unused trailing slots are zero. |
| `active_rounds` | Number of time steps with at least one real event. |
| `maximum_round_activity` | Largest event count in one time step. |
| `distance`, `rounds` | Permitted numeric metadata. |
| `basis` | Permitted `X`/`Z` category, encoded inside each train-fitted pipeline. |

At time step `t`, only the first `round_widths[t]` syndrome positions are real. The fixed 26-slot activity representation comes from the documented training schema; longer rows raise an error. The exact 33-column order is printed in `run_settings` above.

## 3. Load the two published samples and build aligned features and targets

The loader is called once per named split; sampling is independent within each split. There is no new train/validation partition. The dataset builder may prepare other files internally, but this notebook selects and analyzes only `train` and `validation`. Feature extraction runs in chunks to limit temporary memory. The target uses the same selected row order as its feature frame.

In [ ]:
from router.data import load_surface_code_dataset, sample_split
from router.labels import route_to_correlated

requested_splits = []
samples = {}
full_split_sizes = {}
for split_name in ("train", "validation"):
    requested_splits.append(split_name)
    loaded, loaded_config = load_surface_code_dataset(
        config_path=str(config_path), split_name=split_name
    )
    assert set(loaded) == {split_name}
    assert loaded_config["dataset_revision"] == DATASET_REVISION
    full_split_sizes[split_name] = len(loaded[split_name])
    samples[split_name] = sample_split(
        loaded, split_name=split_name,
        sample_size=SAMPLE_SIZES[split_name], seed=SEED,
    )
    assert len(samples[split_name]) == SAMPLE_SIZES[split_name]
assert requested_splits == ["train", "validation"]
print("Requested splits:", requested_splits)
for split_name in requested_splits:
    print(f"{split_name}: {full_split_sizes[split_name]:,} published rows; "
          f"{len(samples[split_name]):,} sampled rows")

In [ ]:
from tqdm.auto import tqdm

X = {}
y = {}
for split_name in requested_splits:
    sample = samples[split_name]
    chunks = []
    for start in tqdm(range(0, len(sample), 5_000),
                      desc=f"Extracting {split_name} features", unit="chunk"):
        stop = min(start + 5_000, len(sample))
        chunks.append(pd.DataFrame(extract_features(sample.select(range(start, stop)))))
    X[split_name] = pd.concat(chunks, ignore_index=True)
    y[split_name] = pd.Series(
        route_to_correlated(
            sample["label"], sample["pymatching"], sample["correlated_matching"]
        ).astype(int),
        index=X[split_name].index,
        name="y_router",
    )
    assert tuple(X[split_name].columns) == expected_columns
    assert X[split_name].shape == (SAMPLE_SIZES[split_name], len(expected_columns))
    assert y[split_name].index.equals(X[split_name].index)
    assert not (set(X[split_name].columns) & FORBIDDEN_COLUMNS)
    validate_features(X[split_name], FEATURE_COLUMNS)
X_train, X_validation = X["train"], X["validation"]
y_train, y_validation = y["train"], y["validation"]
assert set(y_train.unique()) == {0, 1}
assert set(y_validation.unique()).issubset({0, 1})
print("Train/validation shapes:", X_train.shape, X_validation.shape)
print("Train rescue prevalence:", round(float(y_train.mean()), 6))
print("Validation rescue prevalence:", round(float(y_validation.mean()), 6))

## 4. Fit the four candidates on train only

`fit_router()` constructs separate pipelines for Dummy, Logistic Regression, Random Forest, and XGBoost. Each pipeline has an explicit `FEATURE_COLUMNS` allowlist and `remainder="drop"`. Imputation, numeric scaling where configured, and basis one-hot encoding are fitted inside each training pipeline. `predict_rescue_proba()` applies each fitted pipeline to validation. Progress bars show feature chunks and completed model fits/scores; the training bar advances after each model finishes. These are rescue scores in `[0, 1]`; calibration has not been checked.

In [ ]:
models = {}
with tqdm(MODEL_NAMES, desc="Training routers", unit="model") as progress:
    for name in progress:
        progress.set_description(f"Training {name}")
        models[name] = fit_router(
            name, X_train, y_train, allowed_columns=FEATURE_COLUMNS,
            config=model_config,
        )
scores = {}
with tqdm(MODEL_NAMES, desc="Scoring validation", unit="model") as progress:
    for name in progress:
        progress.set_description(f"Scoring {name}")
        scores[name] = predict_rescue_proba(models[name], X_validation)
assert tuple(models) == tuple(scores) == MODEL_NAMES
for name in MODEL_NAMES:
    pipe = models[name]
    score = np.asarray(scores[name])
    assert pipe.allowed_columns_ == list(FEATURE_COLUMNS)
    assert pipe.named_steps["prep"].remainder == "drop"
    assert score.shape == (len(X_validation),)
    assert np.isfinite(score).all() and ((score >= 0) & (score <= 1)).all()
print("All four train-fitted pipelines produced aligned validation scores.")

## 5. Compare rescue prediction on validation

Average precision uses the full continuous rescue scores via scikit-learn's `average_precision_score`. Precision, recall, and F1 use the **provisional 0.5 threshold**; undefined values are set to zero. Rescue prevalence gives the baseline context for this minority target. Target metrics alone do not measure the quality of the final decoder choice.

In [ ]:
from router.evaluate import rescue_metrics

prevalence = float(y_validation.mean())
print(f"Validation rescue prevalence: {prevalence:.6f} "
      f"({int(y_validation.sum()):,}/{len(y_validation):,})")
target_rows = [
    {"model": name, **rescue_metrics(y_validation, scores[name], threshold=THRESHOLD)}
    for name in MODEL_NAMES
]
target_table = pd.DataFrame(target_rows).set_index("model")
print(f"Validation rescue metrics; provisional threshold = {THRESHOLD:.1f}")
display(target_table.round(6))

## 6. Compare final decoder choices on the same validation rows

At the **provisional 0.5 threshold**, `choose_correlated()` turns each rescue score into a decoder choice; `routed_prediction()` selects the stored decoder output. The oracle chooses correlated matching exactly on rescue rows and keeps PyMatching in ties (both correct or both wrong). It is a reference bound, not a deployable router. All rows in this table use the same denominator. Logical error rate is the fraction of final predictions that disagree with `label`; usage is the fraction sent to correlated matching. Rescue capture and harmful switches are counts.

In [ ]:
from router.decoders import outcome_masks, summarize_outcomes
from router.evaluate import system_metrics
from router.routing import choose_correlated, routed_prediction

validation = samples["validation"]
label = np.asarray(validation["label"])
pm = np.asarray(validation["pymatching"])
cm = np.asarray(validation["correlated_matching"])
masks = outcome_masks(label, pm, cm)
assert np.array_equal(y_validation.to_numpy(dtype=bool), masks["rescue"])
assert sum(int(mask.sum()) for mask in masks.values()) == len(validation)
choices = {
    "always_pymatching": np.zeros(len(validation), dtype=bool),
    "always_correlated_matching": np.ones(len(validation), dtype=bool),
}
choices.update({
    name: choose_correlated(scores[name], THRESHOLD) for name in MODEL_NAMES
})
choices["two_decoder_oracle"] = masks["rescue"].copy()
for name, decision in choices.items():
    assert decision.shape == (len(validation),)
    assert np.array_equal(
        routed_prediction(decision, pm, cm), np.where(decision, cm, pm)
    )
system_rows = [
    {"router": name, **system_metrics(label, pm, cm, decision)}
    for name, decision in choices.items()
]
system_table = pd.DataFrame(system_rows).set_index("router")
summary = summarize_outcomes(label, pm, cm)
assert np.isclose(system_table.loc["always_pymatching", "logical_error_rate"],
                  summary["pymatching_ler"])
assert np.isclose(system_table.loc["always_correlated_matching", "logical_error_rate"],
                  summary["correlated_matching_ler"])
assert np.isclose(system_table.loc["two_decoder_oracle", "logical_error_rate"],
                  summary["oracle_ler"])
assert system_table.loc["two_decoder_oracle", "rescues_captured"] == summary["rescue"]
assert system_table.loc["two_decoder_oracle", "harmful_switches"] == 0
print(f"Validation system metrics; provisional threshold = {THRESHOLD:.1f}")
display(system_table.round(6))

## 7. Check the feature boundary and train-only preprocessing

The model code already checks this boundary. These notebook assertions make the run auditable: forbidden fields are rejected, every pipeline drops unspecified columns, and validation-only category and numeric values are transformed without changing fitted training state. The perturbation is temporary and is never used for reported scores. The validation score arrays and decoder outcomes retain the selected sample order.

In [ ]:
forbidden_probe = X_train.iloc[:1].assign(label=0)
try:
    validate_features(forbidden_probe, FEATURE_COLUMNS)
except ValueError as exc:
    assert "forbidden" in str(exc)
else:
    raise AssertionError("A label column entered model features")

for name, pipe in models.items():
    prep = pipe.named_steps["prep"]
    assert prep.remainder == "drop"
    transformer_columns = [
        column for _, _, columns in prep.transformers
        for column in columns if isinstance(column, str)
    ]
    assert set(transformer_columns) == set(FEATURE_COLUMNS)
    baseline = prep.transform(X_train.iloc[:1])
    probe = X_validation.iloc[:1].copy()
    probe.loc[:, "basis"] = "validation_only_basis"
    probe.loc[:, "detection_event_count"] = 1_000_000
    prep.transform(probe)
    assert np.array_equal(prep.transform(X_train.iloc[:1]), baseline)
    categories = prep.named_transformers_["cat"].named_steps["onehot"].categories_[0]
    assert "validation_only_basis" not in categories
    assert np.array_equal(scores[name], predict_rescue_proba(pipe, X_validation))
assert requested_splits == ["train", "validation"]
print("Feature boundary, train-only transformation, score alignment, and split checks passed.")

## 8. Interpret and hand off aggregate results

Copy the exact printed run settings and the two aggregate tables to [the Week 2 classical note](../results/model_scores/week2_classical.md) after the Colab run. Record any model or hyperparameter choice made after inspecting validation, including its rationale. The fixed model settings above are the starting comparison; this notebook does not select an operating model or threshold.

The 0.5 results are a diagnostic. A threshold sweep belongs in notebook 04; the untouched test comparison belongs in notebook 05. These scores are development estimates from published within-experiment splits and do not establish performance on unseen experiments. Rescue probabilities are not described as calibrated unless calibration is separately checked. Keep shot-level rows, feature tables, caches, and fitted models out of Git.